# Introduction to Machine Learning: Illustrated Principles
## Notebook to accompany Chapter 8: Neural Networks

## Overview

This notebook set deals with the [MNIST digit dataset](https://archive.ics.uci.edu/dataset/683/mnist+database+of+handwritten+digits) dataset.

The goal is to classify images into which digit was written.


In [ ]:
# For many systems, for this small network, using more than 1 CPU core will slow things down.
# These lines (and those below) will usually limit numpy to 1 core, which is often faster for this small example.
%env MKL_NUM_THREADS=1
%env NUMEXPR_NUM_THREADS=1
%env OMP_NUM_THREADS=1
%env OPENBLAS_NUM_THREADS=1

## These are the only libraries you should need
import numpy as np
import matplotlib.pyplot as plt

# These are just to help with interactive plotting
from IPython import display
import time
import os

# The following lines are often also needed to limit to 1 CPU core.
from threadpoolctl import threadpool_info
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
os.environ["MKL_ENABLE_INSTRUCTIONS"] = "MAX"
os.environ["MKL_DYNAMIC"] = "FALSE"

### Dataset

Each example is a 14-by-14 gray-scale image (downsampled from a 28-by-28 images) of a handwritten digit.  The goal is to classify an example into one of the ten digits, 0 through 9.  For this notebook, we will focus on distinguishing 7s from 9s, the two hardest classes to separate.

The code below loads the data and separates out only the 7s and 9s, letting $y=0$ represent a 7 and $y=1$ represent a 9.  Each $x$ has $14\times 14 = 196$ features, each one the intensity of one pixel in the image.

The code below loads in a training dataset (`train`), a validation dataset (`valid`), and a testing dataset (`test`).  Each has a `.X` and `.Y` field: numpy arrays of the $X$ and $Y$ matrices.

We are using only 1000 examples for training and for validation.  This simulates a harder problem and makes the effect of regularization more apparent.  You may change it below, but the computational time will increase and the effects of regularization will not be as apparent.

In [ ]:
import sys; sys.path.insert(0, '../..') # path to dataset.py (adjust if you have moved this file or dataset.py)
from dataset import loaddataset
import copy

#adjust the directory below if you have moved either the dataset files or this notebook
datasetdir = '../../datasets/' 

train = loaddataset(datasetdir+'mnist-train')
test = loaddataset(datasetdir+'mnist-test')

## select down only 7s and 9s and remap to labels 0 and 1
labels = (7,9)
trainuse = (train.Y==labels[0]) | (train.Y==labels[1])
train.X = train.X[trainuse,:]
train.Y = (train.Y[trainuse]-labels[0])//(labels[1]-labels[0])
train.ycats = [train.ycats[labels[0]],train.ycats[labels[1]]]
testuse = (test.Y==labels[0]) | (test.Y==labels[1])
test.X = test.X[testuse,:]
test.Y = (test.Y[testuse]-labels[0])//(labels[1]-labels[0])
test.ycats = [test.ycats[labels[0]],test.ycats[labels[1]]]

# downsample images from 28-by-28 to 14-by-14 [You can remove this to work with higher-resolution images, but the computations will take longer.]
train.X = np.reshape(np.reshape(train.X,(-1,14,2,14,2)).sum(axis=(2,4)),(-1,14*14))/4
test.X = np.reshape(np.reshape(test.X,(-1,14,2,14,2)).sum(axis=(2,4)),(-1,14*14))/4

## split train into train and valid
ntrain = 1000
nvalid = 1000
rs = np.random.RandomState(seed=111826)
valid = copy.copy(train)
X = train.X
Y = train.Y
p = rs.permutation(train.X.shape[0])
train.X = X[p[0:ntrain]]
train.Y = Y[p[0:ntrain]]
valid.X = X[p[ntrain:nvalid+ntrain]]
valid.Y = Y[p[ntrain:nvalid+ntrain]]

### Data Pre-processing
As common, we do z-score normalization (see previous chapters' notebooks or chapter 9): replace each feature with the number of standard deviations the raw feature is from the mean in the training test.

In [ ]:
# z-score normalize
mu = np.mean(train.X,axis=0)
stddev = np.std(train.X,axis=0)
stddev[stddev==0] = 1 # features with no variance go unchanged
train.X = ((train.X-mu)/stddev).astype(np.float32)
valid.X = ((valid.X-mu)/stddev).astype(np.float32)
test.X = ((test.X-mu)/stddev).astype(np.float32)

## Question 1: **Neural Network Training**
---
> Write `nneval` and `trainneuralnet`
>
> Note that `train.Y` has values 0 and 1, but our derivation of the algorithms expects -1 and +1 for $y$.  You should adjust `Y` inside the functions to match the derivations.  The internal nodes should all use the hyperbolic tangent (np.tanh) as the nonlinearity.
> 
>  You need to write the two functions below (plus any more you would like to add to help): `nneval` and `trainneuralnet`.  The first takes an array/matrix of X vectors and the weights from a neural network and returns a vector of discriminant values (should be numbers between -infinity and +infinity).  The second takes a data set (Xs and Ys), the number of hidden units, and the lambda value (for l2-regularization), and returns the weights, Wts.  Wts[0] are the weights from the input to the first hidden layer.  Wts[1] are the weights from the first hidden layer to the second hidden layer.  Wts[2] are the weights from the second hidden layer to the output.
> 
> A few notes:
> - **Starting Weights**: The code supplied randomly selects the weights near zero.  This is called [Xavier initialization](https://medium.com/@shivansh.k.sahu/from-zero-to-xavier-understanding-weight-initialization-in-neural-networks-d2d71a28b232).  But for the purposes of the assignment, you can just accept this is a good way to initialize neural network weights.
> - **Offset Terms**: Each layer should have an "offset" or "intercept" unit (to supply a 1 to the next layer), except the output layer.
> - **Batch Updates**: For a problem this small, use batch updates.  That is, the step is based on the sum of the gradients for each data point in the training set.
> - **Step Size**: [This tutorial](https://www.digitalocean.com/community/tutorials/intro-to-optimization-momentum-rmsprop-adam) describes a number of methods to adaptively control $\eta$ for fast convergence.  You don't need to understand any of them; however, without them, convergence to good solutions on this problem can be quite slow.  Therefore, *use RMSprop*: the code below has a simple version of RMSprop that is sufficient for this assignment.  You need to supply the code that calculates `sumofgrad2` which should be the sum of the square of each element of the *full* gradient (the squared length of the gradient including the regularization term).  (for debugging, feel free to use a constant $\eta$ initially). 
> - **Stopping Criterion**: To determine when to stop, check the loss (full loss, including the regularization term) function every 10 iterations.  If it has not improved by at least $10^{-8}$ over those 10 iterations, stop.
> - **Regularization**: You should penalize (from the regularization) all of the weights, even those coming out of offset units.  While it makes sense sometimes not to penalize the ones for the constant $1$ units, you'll find this easier if you just penalize them all.
> 
> Tips that might help:
> - Display the loss function's value every 10 iterations (or so).  It should be getting smaller.  If not, your gradient is not pointing in the right direction.
> - The most common mistake is to subtly have the wrong loss function or the wrong gradient.  The gradient is the gradient of the entire loss function, including the regularization term.  A small mistake will cause the loss function to initially get better, but then to get worse.
> - The smaller $\lambda$ is and the more units, the more difficult (longer) the optimization will be.
> - Write a function to do forward propagation and one to do backward propagation.  Write a function to evaluate the loss function.  In general, break things up to keep things straight.
> - Processing the entire batch at once is more efficient in numpy than using a loop over the examples.  Use numpy broadcasting to avoid loops where possible.
> - Create a small dataset from the examples in the book.  Check that you get the same results for forward propagation.  Check you get the same results for backward propagation.  Check that you get the same results for the gradients.
> - You can reduce the stopping criterion (from $10^{-8}$ to something closer to 1) temporarily to make it finish faster.
---

In [ ]:
### FEEL FREE TO ADD HELPER FUNCTIONS

def nneval(X,Wts):
    # X is m-by-n.  Should return an m-dimensional vector of the discrimanant values for each input row
    # your code here
    pass
    

# Your functions need only work for neural networks of exactly 3 layers of weights
# This training function has a single scalar parameter, nhid, to indicate the number of
# hidden units.  This is the number in the first hidden layer.  The second hidden layer will have 1/2 this number
def trainneuralnet(X,Y,nhid,lam,printinfo=False):
    # The number of examples (m) and number of input dimensions (n) of the input data
    (m,n) = X.shape
    Y = 2*Y[:,None]-1 # to convert from 0,1 to -1,+1
    
    W1 = (np.random.rand(nhid,n+1)*2-1)*np.sqrt(6.0/(n+nhid+1)).astype(np.float32) # weights to each hidden unit from the inputs (plus the added offset unit)
    W2 = (np.random.rand(nhid//2,nhid+1)*2-1)*np.sqrt(6.0/(nhid+nhid//2+1)).astype(np.float32)
    W3 = (np.random.rand(1,nhid//2+1)*2-1)*np.sqrt(6.0/(nhid//2+2)).astype(np.float32) # weights to the single output unit from the hidden units (plus the offset unit)
    W1[:,0] = 0 # good initializations for the constant terms
    W2[:,0] = 0
    W3[:,0] = 0
    
    Wts = [W1,W2,W3] # I put them together in a list, but you can use a tuple too

    # initialization for RMSprop
    Eg2=1

    # your code here

    # the block of code below should be run just before each gradient step to calculate the size of eta for that step
    # the variable sumofgrad2 should be the sum of the squares of each element in the gradient
    '''
    Eg2 = 0.9*Eg2 + 0.1*sumofgrad2
    eta = 0.01/(np.sqrt((1e-10+Eg2)))
    '''
        
    return Wts 

In [ ]:
%%time
# Use this cell (or others you add) to check your network
# I would debug on simple examples you create yourself (trying to understand what happens with
#  the full 256-dimensional data is hard)

#an example of training on the USPS data with 16/8 hidden units and lambda=0.00001, takes about 5000 iterations and about 10 seconds for the solutions
Wts = trainneuralnet(train.X,train.Y,16,1e-5,True)

## Question 2: Validation Errors
---
> The code below compares the validation error for different numbers of hidden units and regularization strengths.
> It also plots the logistic regression validation error (for comparison).
> Run the code and answer the question that follows.
>
> The solutions take about 20 minutes to generate the full plot.  You may wish to reduce the number of restarts or the convergence criterion for the training while debugging your solutions.
---

In [ ]:
## Code added to do logistic regression, just for a comparison
def learnlogreg_sklearn(X,Y,regstr=0.0,penalizeb=False):
    X = addones(X)
    from sklearn.linear_model import LogisticRegression as lr
    from sklearn import __version__ as skver
    if tuple(map(int,skver.split('.'))) <= (1,8,0):
        opts = dict(tol=1e-6,max_iter=100000,
                    penalty=(None if regstr==0.0 else 'l2'),
                    C = (np.inf if regstr==0.0 else 0.5/regstr),
                    solver='newton-cg')
    else:
        opts = dict(tol=1e-6,max_iter=100000,l1_ratio=0.0,
                    C= (np.inf if regstr==0.0 else 0.5/regstr),
                    solver='newton-cg')
        
    if regstr==0.0:
        C = np.inf
    else:
        C = 2.0/regstr # ???

    Y = (Y>0.5).astype(int)
    if penalizeb:
        lrres = lr(fit_intercept=False,**opts).fit(X,Y)
        w = lrres.coef_[0,:]
    else:
        lrres = lr(fit_intercept=True,**opts).fit(X[:,1:],Y)
        w = np.hstack((lrres.intercept_,lrres.coef_[0,:]))
    return w

def predictlogreg(X,w):
    return addones(X)@w

In [ ]:
# Code to update the plot as new results are generated
def setupfig():
    f = plt.figure()
    f.set_size_inches(8,8)
    ax = f.add_subplot(111)
    plt.ion()
    f.canvas.draw()
    return (f,ax)

def plotit(lams,nhiddens,erates,f,ax):
    ax.clear()
    for i in range(nhiddens.shape[0]):
        ax.plot(lams,erates[:,i],'*-')
    ax.set_yscale('log',subs=[1,2,3,4,5,6,7,8,9])
    ax.set_yticks([0.1,0.01])
    ax.set_xscale('log')
    f.canvas.draw()
    ax.set_xlabel('lambda')
    ax.set_ylabel('validation error rate')
    ax.legend([(('# hidden units = '+str(x)+','+str(x//2)) if x>0 else 'logistic regression') for x in nhiddens])
    display.display(f)
    display.clear_output(wait=True)
    

In [ ]:
%%time
nrestarts = 3 # change to a smaller value to speed up plot generation for debugging

def errorrate(Y,predy):
    agree = ((Y>0.5).astype(int)*2-1)*predy[:,0]
    return (agree<=0).mean()
    
def multirestart(trainX,trainY,nhid,lam,ntries):
    besterrsc = 2.0
    for i in range(ntries):
        Wts = trainneuralnet(trainX,trainY,nhid,lam)
        errsc = errorrate(trainY,nneval(trainX,Wts))
        if errsc<besterrsc:
            returnWts = Wts
            besterrsc = errsc
    return returnWts

nhiddens = np.array([0,2,8,16])
lams = np.logspace(-2.5,2.5,9)
erates = np.empty([lams.shape[0],nhiddens.shape[0]])
erates[:,:] = np.nan

(f,ax) = setupfig()

for ni, nhid in enumerate(nhiddens):
    for li, lam in reversed(list(enumerate(lams))):
        if nhid==0:
            w = learnlogreg_sklearn(train.X,train.Y,lam)
            predy = predictlogreg(valid.X,w)[:,np.newaxis]
        else:
            Wts = multirestart(train.X,train.Y,nhid,lam,nrestarts)
            predy = nneval(valid.X,Wts)
        erates[li,ni] = errorrate(valid.Y,predy)
        
        plotit(lams,nhiddens,erates,f,ax)

---
> Explain the plot above, particularly how the validation error varies by regularization strength and by number of hidden units.
> 
> What does this tell you about the data and about applying neural networks to the problem?
> 
> What about this agrees with the book and what aspects do not?
---

## Your answer here

## Question 3: Testing Error
---
> Use the best model according to the validation loss in Question 2 and report its error rate on the testing data.
> (You can use the `erates` variable from above, but you will probably need to retrain the network because the weights were not saved.)
---

In [ ]:
# your code here